**Baricenter Analsysis**
In this notebook we use information from previous geographia analysis.
Working directory (authoritative): C:\Python\trade\geo Authoritative data artifacts: Centroids (single source of truth) C:\Python\trade\geo\country_centroids_augmented.csv Columns include: code (ISO3 / trade country code, incl. historic + territories) lat, lon (WGS84) resolution_method, resolution_path, assigned_from These centroids already resolve: Natural Earth countries historic jurisdictions (via chaining) subnational territories (many via manual coordinates) Origin–Destination distance matrix C:\Python\trade\geo\OD_Matrix.csv Long format: origin, destination, distance_km Distances are haversine, symmetric, and consistent with the centroids above. Basemap (for visualization only) C:\Python\trade\geo\ne_110m_admin_0_countries\ne_110m_admin_0_countries.shp CRS: EPSG:4326 (project to EPSG:3857 for maps). Antarctica (ISO_A3 = ATA) should be excluded from plots.

**1.  Global Trade**

In [2]:
import os
import duckdb
import pandas as pd
import numpy as np

# -------------------------
# Authoritative paths
# -------------------------
GEO_DIR = r"C:\Python\trade\geo"
CENTROIDS_CSV = os.path.join(GEO_DIR, "country_centroids_augmented.csv")

# Trade files directory (adjust if your parquet files are elsewhere)
TRADE_DIR = r"C:\Python\trade\dataverse_files"   # <-- change if needed

OUT_DIR = os.path.join(GEO_DIR, "barycenter")
OUT_FILE = os.path.join(OUT_DIR, "barycenter_imports_exports_1977_2022.csv")

START_YEAR, END_YEAR = 1977, 2022

# -------------------------
# Load centroids (single source of truth)
# -------------------------
centroids = pd.read_csv(CENTROIDS_CSV, usecols=["code", "lat", "lon"])
centroids["code"] = centroids["code"].astype(str)

# Basic sanity: drop rows with missing coords (do not impute)
centroids = centroids.dropna(subset=["lat", "lon"]).copy()

# -------------------------
# Helper: weighted mean (safe)
# -------------------------
def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    w = weights.to_numpy(dtype=float)
    v = values.to_numpy(dtype=float)
    s = np.nansum(w)
    if s == 0 or np.isnan(s):
        return np.nan
    return float(np.nansum(v * w) / s)

# -------------------------
# Main loop
# -------------------------
os.makedirs(OUT_DIR, exist_ok=True)
con = duckdb.connect()

rows = []

for year in range(START_YEAR, END_YEAR + 1):
    fpath = os.path.join(TRADE_DIR, f"S2_{year}.parquet")
    if not os.path.exists(fpath):
        # Keep a record if you want; otherwise skip silently
        continue

    # Aggregate with DuckDB for speed
    # Exports: sum by exporter
    exports_df = con.execute(f"""
        SELECT exporter AS code, SUM(value_final) AS exports_value_final
        FROM read_parquet('{fpath}')
        GROUP BY exporter
    """).df()

    # Imports: sum by importer
    imports_df = con.execute(f"""
        SELECT importer AS code, SUM(value_final) AS imports_value_final
        FROM read_parquet('{fpath}')
        GROUP BY importer
    """).df()

    # Ensure code is str
    exports_df["code"] = exports_df["code"].astype(str)
    imports_df["code"] = imports_df["code"].astype(str)

    # Merge with fixed centroids (no geographic modifications)
    ex = exports_df.merge(centroids, on="code", how="left")
    im = imports_df.merge(centroids, on="code", how="left")

    # Unmatched codes (trade present but no centroid)
    ex_unmatched = ex["lat"].isna() | ex["lon"].isna()
    im_unmatched = im["lat"].isna() | im["lon"].isna()

    n_ex_total = len(ex)
    n_im_total = len(im)
    n_ex_matched = int((~ex_unmatched).sum())
    n_im_matched = int((~im_unmatched).sum())
    n_ex_unmatched = int(ex_unmatched.sum())
    n_im_unmatched = int(im_unmatched.sum())

    # Drop unmatched for barycenter calculation
    exm = ex.loc[~ex_unmatched].copy()
    imm = im.loc[~im_unmatched].copy()

    # Totals (over matched set used for barycenter)
    total_exports_matched = float(exm["exports_value_final"].sum()) if len(exm) else 0.0
    total_imports_matched = float(imm["imports_value_final"].sum()) if len(imm) else 0.0

    # Barycenters (simple weighted mean in lat/lon)
    lat_exports = weighted_mean(exm["lat"], exm["exports_value_final"]) if len(exm) else np.nan
    lon_exports = weighted_mean(exm["lon"], exm["exports_value_final"]) if len(exm) else np.nan

    lat_imports = weighted_mean(imm["lat"], imm["imports_value_final"]) if len(imm) else np.nan
    lon_imports = weighted_mean(imm["lon"], imm["imports_value_final"]) if len(imm) else np.nan

    # Optional: list unmatched codes (truncated) for auditing
    # Keep them as a semicolon-delimited string; cap length to avoid huge CSV cells
    ex_unmatched_codes = ex.loc[ex_unmatched, "code"].unique().tolist()
    im_unmatched_codes = im.loc[im_unmatched, "code"].unique().tolist()

    def truncate_codes(codes, max_codes=50):
        if not codes:
            return ""
        if len(codes) <= max_codes:
            return ";".join(codes)
        return ";".join(codes[:max_codes]) + f";...(plus {len(codes)-max_codes} more)"

    rows.append({
        "year": year,

        "lat_exports": lat_exports,
        "lon_exports": lon_exports,
        "lat_imports": lat_imports,
        "lon_imports": lon_imports,

        "total_exports_value_final_matched": total_exports_matched,
        "total_imports_value_final_matched": total_imports_matched,

        "n_exporter_nodes_total": n_ex_total,
        "n_importer_nodes_total": n_im_total,
        "n_exporter_nodes_matched": n_ex_matched,
        "n_importer_nodes_matched": n_im_matched,
        "n_exporter_nodes_unmatched": n_ex_unmatched,
        "n_importer_nodes_unmatched": n_im_unmatched,

        "unmatched_exporter_codes_sample": truncate_codes(ex_unmatched_codes),
        "unmatched_importer_codes_sample": truncate_codes(im_unmatched_codes),
    })

# -------------------------
# Write output (overwrite)
# -------------------------
out = pd.DataFrame(rows).sort_values("year")
out.to_csv(OUT_FILE, index=False)

print(f"Wrote: {OUT_FILE}")
print(f"Years processed: {out['year'].nunique()} (from {out['year'].min()} to {out['year'].max()})")
print("If unmatched codes exist, inspect the *_codes_sample columns for auditing.")


Wrote: C:\Python\trade\geo\barycenter\barycenter_imports_exports_1977_2022.csv
Years processed: 46 (from 1977 to 2022)
If unmatched codes exist, inspect the *_codes_sample columns for auditing.


***Global Trade Maps***

In [6]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString
from pyproj import CRS

# ------------------------------------------------------------
# Paths (GLOBAL barycenters already computed)
# ------------------------------------------------------------
GEO_DIR = r"C:\Python\trade\geo"
SHAPEFILE = os.path.join(GEO_DIR, r"ne_110m_admin_0_countries\ne_110m_admin_0_countries.shp")

BARY_DIR = os.path.join(GEO_DIR, "barycenter")
GLOBAL_CSV = os.path.join(BARY_DIR, "barycenter_imports_exports_1977_2022.csv")

OUT_IMPORTS_PNG = os.path.join(BARY_DIR, "barycenter_global_imports_mediterranean.png")
OUT_IMPORTS_PDF = os.path.join(BARY_DIR, "barycenter_global_imports_mediterranean.pdf")

OUT_EXPORTS_PNG = os.path.join(BARY_DIR, "barycenter_global_exports_mediterranean.png")
OUT_EXPORTS_PDF = os.path.join(BARY_DIR, "barycenter_global_exports_mediterranean.pdf")

# ------------------------------------------------------------
# Load basemap and remove Antarctica
# ------------------------------------------------------------
world = gpd.read_file(SHAPEFILE)
if "ISO_A3" in world.columns:
    world = world[world["ISO_A3"] != "ATA"].copy()
if world.crs is None:
    world = world.set_crs(epsg=4326)

# ------------------------------------------------------------
# Load GLOBAL barycenters
# ------------------------------------------------------------
df = pd.read_csv(GLOBAL_CSV).sort_values("year")
df = df.dropna(subset=["lat_exports", "lon_exports", "lat_imports", "lon_imports"]).copy()

exports = gpd.GeoDataFrame(
    df[["year", "lat_exports", "lon_exports"]],
    geometry=[Point(xy) for xy in zip(df["lon_exports"], df["lat_exports"])],
    crs="EPSG:4326",
).sort_values("year")

imports = gpd.GeoDataFrame(
    df[["year", "lat_imports", "lon_imports"]],
    geometry=[Point(xy) for xy in zip(df["lon_imports"], df["lat_imports"])],
    crs="EPSG:4326",
).sort_values("year")

exports_path = gpd.GeoDataFrame(
    geometry=[LineString(exports.geometry.tolist())],
    crs="EPSG:4326"
)

imports_path = gpd.GeoDataFrame(
    geometry=[LineString(imports.geometry.tolist())],
    crs="EPSG:4326"
)

# ------------------------------------------------------------
# SAME Mediterranean projection (as USA map)
# ------------------------------------------------------------
crs_med = CRS.from_proj4("+proj=aeqd +lat_0=35 +lon_0=18 +datum=WGS84 +units=m +no_defs")

world_p = world.to_crs(crs_med)
exports_p = exports.to_crs(crs_med)
imports_p = imports.to_crs(crs_med)
exports_path_p = exports_path.to_crs(crs_med)
imports_path_p = imports_path.to_crs(crs_med)

# ------------------------------------------------------------
# SAME zoom window logic as USA map:
# derive extent from GLOBAL points themselves (then apply same buffer)
# This reproduces the exact global "Mediterranean zoom" frame you used.
# ------------------------------------------------------------
minx, miny, maxx, maxy = gpd.GeoSeries(
    pd.concat([exports_p.geometry, imports_p.geometry]),
    crs=crs_med
).total_bounds

buffer = 1.2e6  # must match your USA/global figure buffer
X_LIM = (minx - buffer, maxx + buffer)
Y_LIM = (miny - buffer, maxy + buffer)

# ------------------------------------------------------------
# Year labels: SAME as USA map, same font size
# ------------------------------------------------------------
LABEL_YEARS = {1977, 1980, 1985, 1994, 2001, 2009, 2018, 2023}

def add_year_labels(ax, gdf_sorted, dx=20000, dy=20000, fontsize=5):
    for _, r in gdf_sorted.iterrows():
        y = int(r["year"])
        if y in LABEL_YEARS:
            x0, y0 = r.geometry.x, r.geometry.y
            ax.text(
                x0 + dx, y0 + dy,
                str(y),
                fontsize=fontsize,
                color="black",
                alpha=0.8,
                zorder=10
            )

# ------------------------------------------------------------
# Render function (same as USA map)
# ------------------------------------------------------------
def render_single_map(points_p, path_p, color, footer_label, out_png, out_pdf):
    fig, ax = plt.subplots(figsize=(10, 6))

    world_p.plot(ax=ax, linewidth=0.4, edgecolor="white", color="lightgray", zorder=1)
    path_p.plot(ax=ax, linewidth=1.1, color=color, alpha=0.28, zorder=3)
    points_p.plot(ax=ax, color=color, markersize=8, zorder=4)

    add_year_labels(ax, points_p.sort_values("year"))

    ax.set_xlim(*X_LIM)
    ax.set_ylim(*Y_LIM)

    ax.text(
        0.02, 0.02,
        footer_label,
        transform=ax.transAxes,
        fontsize=9,
        color="black",
        ha="left",
        va="bottom"
    )

    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)

# ------------------------------------------------------------
# Produce two maps: GLOBAL imports-only and exports-only
# ------------------------------------------------------------
render_single_map(
    imports_p,
    imports_path_p,
    "red",
    "Global imports barycenter",
    OUT_IMPORTS_PNG,
    OUT_IMPORTS_PDF
)

render_single_map(
    exports_p,
    exports_path_p,
    "dodgerblue",
    "Global exports barycenter",
    OUT_EXPORTS_PNG,
    OUT_EXPORTS_PDF
)


**USA**

In [7]:
import os
import duckdb
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString
from pyproj import CRS
from matplotlib.lines import Line2D

# ------------------------------------------------------------
# Authoritative geo paths + trade dir (set correctly)
# ------------------------------------------------------------
GEO_DIR = r"C:\Python\trade\geo"
CENTROIDS_CSV = os.path.join(GEO_DIR, "country_centroids_augmented.csv")
SHAPEFILE = os.path.join(GEO_DIR, r"ne_110m_admin_0_countries\ne_110m_admin_0_countries.shp")

TRADE_DIR = r"C:\Python\trade\dataverse_files"   # <-- change if needed
START_YEAR, END_YEAR = 1977, 2022

OUT_DIR = os.path.join(GEO_DIR, "barycenter")
os.makedirs(OUT_DIR, exist_ok=True)

OUT_CSV = os.path.join(OUT_DIR, "barycenter_usa_trade_1977_2022.csv")
OUT_PNG = os.path.join(OUT_DIR, "barycenter_usa_trade_mediterranean_zoom.png")
OUT_PDF = os.path.join(OUT_DIR, "barycenter_usa_trade_mediterranean_zoom.pdf")

USA_CODE = "USA"  # must match your parquet exporter/importer codes

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    w = weights.to_numpy(dtype=float)
    v = values.to_numpy(dtype=float)
    s = np.nansum(w)
    if s == 0 or np.isnan(s):
        return np.nan
    return float(np.nansum(v * w) / s)

def trunc_codes(codes, max_codes=50) -> str:
    codes = list(codes) if codes is not None else []
    if not codes:
        return ""
    if len(codes) <= max_codes:
        return ";".join(codes)
    return ";".join(codes[:max_codes]) + f";...(plus {len(codes)-max_codes} more)"

# ------------------------------------------------------------
# Load centroids (single source of truth)
# ------------------------------------------------------------
centroids = pd.read_csv(CENTROIDS_CSV, usecols=["code", "lat", "lon"])
centroids["code"] = centroids["code"].astype(str)
centroids = centroids.dropna(subset=["lat", "lon"]).copy()

# ------------------------------------------------------------
# Compute USA partner-weighted barycenters per year
# ------------------------------------------------------------
con = duckdb.connect()
rows = []

for year in range(START_YEAR, END_YEAR + 1):
    fpath = os.path.join(TRADE_DIR, f"S2_{year}.parquet")
    if not os.path.exists(fpath):
        continue

    # USA exports: exporter=USA, partner is importer
    usx = con.execute(f"""
        SELECT importer AS partner, SUM(value_final) AS w
        FROM read_parquet('{fpath}')
        WHERE exporter = '{USA_CODE}'
        GROUP BY importer
    """).df()

    # USA imports: importer=USA, partner is exporter
    usm = con.execute(f"""
        SELECT exporter AS partner, SUM(value_final) AS w
        FROM read_parquet('{fpath}')
        WHERE importer = '{USA_CODE}'
        GROUP BY exporter
    """).df()

    usx["partner"] = usx["partner"].astype(str)
    usm["partner"] = usm["partner"].astype(str)

    usxg = usx.merge(centroids, left_on="partner", right_on="code", how="left")
    usmg = usm.merge(centroids, left_on="partner", right_on="code", how="left")

    usx_unmatched = usxg["lat"].isna() | usxg["lon"].isna()
    usm_unmatched = usmg["lat"].isna() | usmg["lon"].isna()

    usx_mat = usxg.loc[~usx_unmatched].copy()
    usm_mat = usmg.loc[~usm_unmatched].copy()

    total_us_exports = float(usx_mat["w"].sum()) if len(usx_mat) else 0.0
    total_us_imports = float(usm_mat["w"].sum()) if len(usm_mat) else 0.0

    lat_us_exports = weighted_mean(usx_mat["lat"], usx_mat["w"]) if len(usx_mat) else np.nan
    lon_us_exports = weighted_mean(usx_mat["lon"], usx_mat["w"]) if len(usx_mat) else np.nan

    lat_us_imports = weighted_mean(usm_mat["lat"], usm_mat["w"]) if len(usm_mat) else np.nan
    lon_us_imports = weighted_mean(usm_mat["lon"], usm_mat["w"]) if len(usm_mat) else np.nan

    rows.append({
        "year": year,
        "lat_us_exports": lat_us_exports,
        "lon_us_exports": lon_us_exports,
        "lat_us_imports": lat_us_imports,
        "lon_us_imports": lon_us_imports,
        "total_us_exports_value_final_matched": total_us_exports,
        "total_us_imports_value_final_matched": total_us_imports,
        "n_export_partners_total": int(len(usxg)),
        "n_import_partners_total": int(len(usmg)),
        "n_export_partners_unmatched": int(usx_unmatched.sum()),
        "n_import_partners_unmatched": int(usm_unmatched.sum()),
        "unmatched_export_partners_sample": trunc_codes(usxg.loc[usx_unmatched, "partner"].unique()),
        "unmatched_import_partners_sample": trunc_codes(usmg.loc[usm_unmatched, "partner"].unique()),
    })

out = pd.DataFrame(rows).sort_values("year")
out.to_csv(OUT_CSV, index=False)
print(f"Wrote: {OUT_CSV} | Years: {out['year'].nunique()}")

# ------------------------------------------------------------
# Plot: SAME projection + SAME zoom logic as your global map
# (Mediterranean-centered AEQD; zoom to barycenter bounds + buffer)
# ------------------------------------------------------------
world = gpd.read_file(SHAPEFILE)
if "ISO_A3" in world.columns:
    world = world[world["ISO_A3"] != "ATA"].copy()
if world.crs is None:
    world = world.set_crs(epsg=4326)

dfp = out.dropna(subset=["lat_us_exports","lon_us_exports","lat_us_imports","lon_us_imports"]).copy()
dfp = dfp.sort_values("year")

exports = gpd.GeoDataFrame(
    dfp[["year","lat_us_exports","lon_us_exports"]].copy(),
    geometry=[Point(xy) for xy in zip(dfp["lon_us_exports"], dfp["lat_us_exports"])],
    crs="EPSG:4326",
).sort_values("year")

imports = gpd.GeoDataFrame(
    dfp[["year","lat_us_imports","lon_us_imports"]].copy(),
    geometry=[Point(xy) for xy in zip(dfp["lon_us_imports"], dfp["lat_us_imports"])],
    crs="EPSG:4326",
).sort_values("year")

exports_line = LineString(list(exports.geometry.values))
imports_line = LineString(list(imports.geometry.values))

exports_path = gpd.GeoDataFrame({"name":["exports_path"]}, geometry=[exports_line], crs="EPSG:4326")
imports_path = gpd.GeoDataFrame({"name":["imports_path"]}, geometry=[imports_line], crs="EPSG:4326")

# Mediterranean-centered projection (same as your global map)
crs_med = CRS.from_proj4("+proj=aeqd +lat_0=35 +lon_0=18 +datum=WGS84 +units=m +no_defs")

world_p = world.to_crs(crs_med)
exports_p = exports.to_crs(crs_med)
imports_p = imports.to_crs(crs_med)
exports_path_p = exports_path.to_crs(crs_med)
imports_path_p = imports_path.to_crs(crs_med)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))

world_p.plot(ax=ax, linewidth=0.4, edgecolor="white", color="lightgray", zorder=1)

imports_path_p.plot(ax=ax, linewidth=1.0, color="red", alpha=0.25, zorder=3)
exports_path_p.plot(ax=ax, linewidth=1.0, color="dodgerblue", alpha=0.25, zorder=4)

imports_p.plot(ax=ax, color="red", markersize=8, zorder=5)
exports_p.plot(ax=ax, color="dodgerblue", markersize=8, zorder=6)

# Legend
legend_elements = [
    Line2D([0], [0], marker="o", color="w", label="USA imports barycenter (partners)",
           markerfacecolor="red", markersize=6),
    Line2D([0], [0], marker="o", color="w", label="USA exports barycenter (partners)",
           markerfacecolor="dodgerblue", markersize=6),
]
ax.legend(handles=legend_elements, loc="lower left", frameon=False, fontsize=9)

# Decade labels in black + endpoints (same labeling logic)
years = dfp["year"].astype(int).tolist()
ymin, ymax = min(years), max(years)
label_years = set(range((ymin // 10) * 10, ymax + 1, 10))
label_years.update([ymin, ymax])

dx, dy = 35000, 35000  # same default as your global map

def label_decades(gdf_sorted):
    for _, r in gdf_sorted.iterrows():
        y = int(r["year"])
        if y in label_years:
            x0, y0 = r.geometry.x, r.geometry.y
            ax.text(x0 + dx, y0 + dy, str(y), fontsize=8, color="black", alpha=0.9, zorder=10)

label_decades(imports_p.sort_values("year"))
label_decades(exports_p.sort_values("year"))

# Zoom: bounds + buffer (same approach)
all_geom = pd.concat([imports_p.geometry, exports_p.geometry], ignore_index=True)
minx, miny, maxx, maxy = gpd.GeoSeries(all_geom, crs=imports_p.crs).total_bounds

buffer = 1.2e6  # same as your global map; adjust if you want tighter crop
ax.set_xlim(minx - buffer, maxx + buffer)
ax.set_ylim(miny - buffer, maxy + buffer)

ax.set_axis_off()
plt.tight_layout()
plt.savefig(OUT_PNG, dpi=300, bbox_inches="tight")
plt.savefig(OUT_PDF, bbox_inches="tight")
plt.close(fig)

print(f"Saved: {OUT_PNG}")
print(f"Saved: {OUT_PDF}")


Wrote: C:\Python\trade\geo\barycenter\barycenter_usa_trade_1977_2022.csv | Years: 46
Saved: C:\Python\trade\geo\barycenter\barycenter_usa_trade_mediterranean_zoom.png
Saved: C:\Python\trade\geo\barycenter\barycenter_usa_trade_mediterranean_zoom.pdf


In [5]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString
from pyproj import CRS

# ------------------------------------------------------------
# Paths (USA barycenters already computed)
# ------------------------------------------------------------
GEO_DIR = r"C:\Python\trade\geo"
SHAPEFILE = os.path.join(GEO_DIR, r"ne_110m_admin_0_countries\ne_110m_admin_0_countries.shp")

BARY_DIR = os.path.join(GEO_DIR, "barycenter")
USA_CSV = os.path.join(BARY_DIR, "barycenter_usa_trade_1977_2022.csv")

OUT_IMPORTS_PNG = os.path.join(BARY_DIR, "barycenter_usa_imports_mediterranean.png")
OUT_IMPORTS_PDF = os.path.join(BARY_DIR, "barycenter_usa_imports_mediterranean.pdf")

OUT_EXPORTS_PNG = os.path.join(BARY_DIR, "barycenter_usa_exports_mediterranean.png")
OUT_EXPORTS_PDF = os.path.join(BARY_DIR, "barycenter_usa_exports_mediterranean.pdf")

# ------------------------------------------------------------
# Load basemap and remove Antarctica
# ------------------------------------------------------------
world = gpd.read_file(SHAPEFILE)
if "ISO_A3" in world.columns:
    world = world[world["ISO_A3"] != "ATA"].copy()
if world.crs is None:
    world = world.set_crs(epsg=4326)

# ------------------------------------------------------------
# Load USA barycenters
# ------------------------------------------------------------
df = pd.read_csv(USA_CSV).sort_values("year")
df = df.dropna(
    subset=["lat_us_exports", "lon_us_exports", "lat_us_imports", "lon_us_imports"]
).copy()

exports = gpd.GeoDataFrame(
    df[["year", "lat_us_exports", "lon_us_exports"]],
    geometry=[Point(xy) for xy in zip(df["lon_us_exports"], df["lat_us_exports"])],
    crs="EPSG:4326",
).sort_values("year")

imports = gpd.GeoDataFrame(
    df[["year", "lat_us_imports", "lon_us_imports"]],
    geometry=[Point(xy) for xy in zip(df["lon_us_imports"], df["lat_us_imports"])],
    crs="EPSG:4326",
).sort_values("year")

exports_path = gpd.GeoDataFrame(
    geometry=[LineString(exports.geometry.tolist())],
    crs="EPSG:4326"
)

imports_path = gpd.GeoDataFrame(
    geometry=[LineString(imports.geometry.tolist())],
    crs="EPSG:4326"
)

# ------------------------------------------------------------
# SAME Mediterranean projection
# ------------------------------------------------------------
crs_med = CRS.from_proj4("+proj=aeqd +lat_0=35 +lon_0=18 +datum=WGS84 +units=m +no_defs")

world_p = world.to_crs(crs_med)
exports_p = exports.to_crs(crs_med)
imports_p = imports.to_crs(crs_med)
exports_path_p = exports_path.to_crs(crs_med)
imports_path_p = imports_path.to_crs(crs_med)

# ------------------------------------------------------------
# SAME zoom window (derived from global barycenter map)
# ------------------------------------------------------------
GLOBAL_BARY_FILE = os.path.join(BARY_DIR, "barycenter_imports_exports_1977_2022.csv")

gdf_global = pd.read_csv(GLOBAL_BARY_FILE).dropna(
    subset=["lat_exports", "lon_exports", "lat_imports", "lon_imports"]
)

global_exports = gpd.GeoDataFrame(
    geometry=[Point(xy) for xy in zip(gdf_global["lon_exports"], gdf_global["lat_exports"])],
    crs="EPSG:4326",
).to_crs(crs_med)

global_imports = gpd.GeoDataFrame(
    geometry=[Point(xy) for xy in zip(gdf_global["lon_imports"], gdf_global["lat_imports"])],
    crs="EPSG:4326",
).to_crs(crs_med)

minx, miny, maxx, maxy = gpd.GeoSeries(
    pd.concat([global_exports.geometry, global_imports.geometry]),
    crs=crs_med
).total_bounds

buffer = 1.2e6
X_LIM = (minx - buffer, maxx + buffer)
Y_LIM = (miny - buffer, maxy + buffer)

# ------------------------------------------------------------
# Year labels: ONLY selected years, smaller font
# ------------------------------------------------------------
LABEL_YEARS = {1977, 1980, 1982, 1983, 1985, 1987, 1990, 1994, 1999, 2001, 2009, 2012, 2015, 2018, 2022}

def add_year_labels(ax, gdf_sorted, dx=20000, dy=20000, fontsize=5):
    for _, r in gdf_sorted.iterrows():
        y = int(r["year"])
        if y in LABEL_YEARS:
            x0, y0 = r.geometry.x, r.geometry.y
            ax.text(
                x0 + dx, y0 + dy,
                str(y),
                fontsize=fontsize,
                color="black",
                alpha=0.8,
                zorder=10
            )

# ------------------------------------------------------------
# Render function
# ------------------------------------------------------------
def render_single_map(points_p, path_p, color, footer_label, out_png, out_pdf):
    fig, ax = plt.subplots(figsize=(10, 6))

    world_p.plot(ax=ax, linewidth=0.4, edgecolor="white", color="lightgray", zorder=1)
    path_p.plot(ax=ax, linewidth=1.1, color=color, alpha=0.28, zorder=3)
    points_p.plot(ax=ax, color=color, markersize=8, zorder=4)

    add_year_labels(ax, points_p.sort_values("year"))

    ax.set_xlim(*X_LIM)
    ax.set_ylim(*Y_LIM)

    ax.text(
        0.02, 0.02,
        footer_label,
        transform=ax.transAxes,
        fontsize=9,
        color="black",
        ha="left",
        va="bottom"
    )

    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)

# ------------------------------------------------------------
# Produce two maps
# ------------------------------------------------------------
render_single_map(
    imports_p,
    imports_path_p,
    "red",
    "USA imports barycenter",
    OUT_IMPORTS_PNG,
    OUT_IMPORTS_PDF
)

render_single_map(
    exports_p,
    exports_path_p,
    "dodgerblue",
    "USA exports barycenter",
    OUT_EXPORTS_PNG,
    OUT_EXPORTS_PDF
)


**China**

In [8]:
import os
import duckdb
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString
from pyproj import CRS
from matplotlib.lines import Line2D

# ------------------------------------------------------------
# CONFIG (edit only TRADE_DIR if needed)
# ------------------------------------------------------------
GEO_DIR = r"C:\Python\trade\geo"
CENTROIDS_CSV = os.path.join(GEO_DIR, "country_centroids_augmented.csv")
SHAPEFILE = os.path.join(GEO_DIR, r"ne_110m_admin_0_countries\ne_110m_admin_0_countries.shp")

TRADE_DIR = r"C:\Python\trade\dataverse_files"  # <-- set to where S2_YYYY.parquet lives
START_YEAR, END_YEAR = 1977, 2022

BARY_DIR = os.path.join(GEO_DIR, "barycenter")
os.makedirs(BARY_DIR, exist_ok=True)

# Use the SAME map window as your global map by reading its barycenter file:
GLOBAL_BARY_FILE = os.path.join(BARY_DIR, "barycenter_imports_exports_1977_2022.csv")

# Output for China
OUT_CSV = os.path.join(BARY_DIR, "barycenter_china_trade_1977_2022.csv")
OUT_PNG = os.path.join(BARY_DIR, "barycenter_china_trade_mediterranean_zoom.png")
OUT_PDF = os.path.join(BARY_DIR, "barycenter_china_trade_mediterranean_zoom.pdf")

# China code (ISO3). We'll validate it exists in your trade files.
FOCUS_CODE = "CHN"

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    w = weights.to_numpy(dtype=float)
    v = values.to_numpy(dtype=float)
    s = np.nansum(w)
    if s == 0 or np.isnan(s):
        return np.nan
    return float(np.nansum(v * w) / s)

def trunc_codes(codes, max_codes=50) -> str:
    codes = list(codes) if codes is not None else []
    if not codes:
        return ""
    if len(codes) <= max_codes:
        return ";".join(codes)
    return ";".join(codes[:max_codes]) + f";...(plus {len(codes)-max_codes} more)"

def first_existing_parquet(trade_dir: str, start: int, end: int):
    for y in range(start, end + 1):
        fp = os.path.join(trade_dir, f"S2_{y}.parquet")
        if os.path.exists(fp):
            return y, fp
    return None, None

# ------------------------------------------------------------
# Load centroids (single source of truth)
# ------------------------------------------------------------
centroids = pd.read_csv(CENTROIDS_CSV, usecols=["code", "lat", "lon"])
centroids["code"] = centroids["code"].astype(str)
centroids = centroids.dropna(subset=["lat", "lon"]).copy()

# ------------------------------------------------------------
# Connect DuckDB + validate that FOCUS_CODE exists (less guess)
# ------------------------------------------------------------
con = duckdb.connect()

y0, fp0 = first_existing_parquet(TRADE_DIR, START_YEAR, END_YEAR)
if fp0 is None:
    raise FileNotFoundError(f"No S2_YYYY.parquet files found in {TRADE_DIR} for {START_YEAR}-{END_YEAR}.")

exists = con.execute(f"""
    SELECT
      SUM(CASE WHEN exporter = '{FOCUS_CODE}' THEN 1 ELSE 0 END) AS as_exporter,
      SUM(CASE WHEN importer = '{FOCUS_CODE}' THEN 1 ELSE 0 END) AS as_importer
    FROM read_parquet('{fp0}')
""").fetchone()

if (exists[0] == 0) and (exists[1] == 0):
    # Show likely candidates by volume (top 20) so you can pick the correct code if different
    top_exporters = con.execute(f"""
        SELECT exporter AS code, SUM(value_final) AS v
        FROM read_parquet('{fp0}')
        GROUP BY exporter
        ORDER BY v DESC
        LIMIT 20
    """).df()

    top_importers = con.execute(f"""
        SELECT importer AS code, SUM(value_final) AS v
        FROM read_parquet('{fp0}')
        GROUP BY importer
        ORDER BY v DESC
        LIMIT 20
    """).df()

    msg = (
        f"FOCUS_CODE='{FOCUS_CODE}' not found as exporter or importer in the first available file "
        f"(S2_{y0}.parquet). Your dataset may use a different code.\n\n"
        f"Top exporters in {y0}:\n{top_exporters.to_string(index=False)}\n\n"
        f"Top importers in {y0}:\n{top_importers.to_string(index=False)}\n"
    )
    raise ValueError(msg)

print(f"Validated: {FOCUS_CODE} appears in S2_{y0}.parquet (exporter count={exists[0]}, importer count={exists[1]}).")

# ------------------------------------------------------------
# Compute partner-weighted barycenters for China per year
# ------------------------------------------------------------
rows = []

for year in range(START_YEAR, END_YEAR + 1):
    fpath = os.path.join(TRADE_DIR, f"S2_{year}.parquet")
    if not os.path.exists(fpath):
        continue

    # Exports barycenter for China: exporter=CHN, partners are importers
    cx = con.execute(f"""
        SELECT importer AS partner, SUM(value_final) AS w
        FROM read_parquet('{fpath}')
        WHERE exporter = '{FOCUS_CODE}'
        GROUP BY importer
    """).df()

    # Imports barycenter for China: importer=CHN, partners are exporters
    cm = con.execute(f"""
        SELECT exporter AS partner, SUM(value_final) AS w
        FROM read_parquet('{fpath}')
        WHERE importer = '{FOCUS_CODE}'
        GROUP BY exporter
    """).df()

    cx["partner"] = cx["partner"].astype(str)
    cm["partner"] = cm["partner"].astype(str)

    cxg = cx.merge(centroids, left_on="partner", right_on="code", how="left")
    cmg = cm.merge(centroids, left_on="partner", right_on="code", how="left")

    cx_unmatched = cxg["lat"].isna() | cxg["lon"].isna()
    cm_unmatched = cmg["lat"].isna() | cmg["lon"].isna()

    cx_mat = cxg.loc[~cx_unmatched].copy()
    cm_mat = cmg.loc[~cm_unmatched].copy()

    total_exports = float(cx_mat["w"].sum()) if len(cx_mat) else 0.0
    total_imports = float(cm_mat["w"].sum()) if len(cm_mat) else 0.0

    lat_exports = weighted_mean(cx_mat["lat"], cx_mat["w"]) if len(cx_mat) else np.nan
    lon_exports = weighted_mean(cx_mat["lon"], cx_mat["w"]) if len(cx_mat) else np.nan

    lat_imports = weighted_mean(cm_mat["lat"], cm_mat["w"]) if len(cm_mat) else np.nan
    lon_imports = weighted_mean(cm_mat["lon"], cm_mat["w"]) if len(cm_mat) else np.nan

    rows.append({
        "year": year,
        "lat_exports": lat_exports,
        "lon_exports": lon_exports,
        "lat_imports": lat_imports,
        "lon_imports": lon_imports,
        "total_exports_value_final_matched": total_exports,
        "total_imports_value_final_matched": total_imports,
        "n_export_partners_total": int(len(cxg)),
        "n_import_partners_total": int(len(cmg)),
        "n_export_partners_unmatched": int(cx_unmatched.sum()),
        "n_import_partners_unmatched": int(cm_unmatched.sum()),
        "unmatched_export_partners_sample": trunc_codes(cxg.loc[cx_unmatched, "partner"].unique()),
        "unmatched_import_partners_sample": trunc_codes(cmg.loc[cm_unmatched, "partner"].unique()),
    })

out = pd.DataFrame(rows).sort_values("year")
out.to_csv(OUT_CSV, index=False)
print(f"Wrote: {OUT_CSV} | Years: {out['year'].nunique()}")

# ------------------------------------------------------------
# Plot: SAME projection + SAME map window as the global map
# ------------------------------------------------------------
world = gpd.read_file(SHAPEFILE)
if "ISO_A3" in world.columns:
    world = world[world["ISO_A3"] != "ATA"].copy()
if world.crs is None:
    world = world.set_crs(epsg=4326)

# Mediterranean-centered projection (same as global map)
crs_med = CRS.from_proj4("+proj=aeqd +lat_0=35 +lon_0=18 +datum=WGS84 +units=m +no_defs")

# Build China barycenter point layers
dfp = out.dropna(subset=["lat_exports","lon_exports","lat_imports","lon_imports"]).copy()

exports_gdf = gpd.GeoDataFrame(
    dfp[["year","lat_exports","lon_exports"]].copy(),
    geometry=[Point(xy) for xy in zip(dfp["lon_exports"], dfp["lat_exports"])],
    crs="EPSG:4326",
).sort_values("year")

imports_gdf = gpd.GeoDataFrame(
    dfp[["year","lat_imports","lon_imports"]].copy(),
    geometry=[Point(xy) for xy in zip(dfp["lon_imports"], dfp["lat_imports"])],
    crs="EPSG:4326",
).sort_values("year")

exports_line = LineString(list(exports_gdf.geometry.values))
imports_line = LineString(list(imports_gdf.geometry.values))

exports_path = gpd.GeoDataFrame({"name":["exports_path"]}, geometry=[exports_line], crs="EPSG:4326")
imports_path = gpd.GeoDataFrame({"name":["imports_path"]}, geometry=[imports_line], crs="EPSG:4326")

# Project layers
world_p = world.to_crs(crs_med)
exports_p = exports_gdf.to_crs(crs_med)
imports_p = imports_gdf.to_crs(crs_med)
exports_path_p = exports_path.to_crs(crs_med)
imports_path_p = imports_path.to_crs(crs_med)

# Compute the global map extent in the SAME projection (so the window matches exactly)
global_df = pd.read_csv(GLOBAL_BARY_FILE).dropna(subset=["lat_exports","lon_exports","lat_imports","lon_imports"]).copy()

global_exports = gpd.GeoDataFrame(
    global_df[["year","lat_exports","lon_exports"]].copy(),
    geometry=[Point(xy) for xy in zip(global_df["lon_exports"], global_df["lat_exports"])],
    crs="EPSG:4326",
).to_crs(crs_med)

global_imports = gpd.GeoDataFrame(
    global_df[["year","lat_imports","lon_imports"]].copy(),
    geometry=[Point(xy) for xy in zip(global_df["lon_imports"], global_df["lat_imports"])],
    crs="EPSG:4326",
).to_crs(crs_med)

all_global_geom = pd.concat([global_exports.geometry, global_imports.geometry], ignore_index=True)
minx, miny, maxx, maxy = gpd.GeoSeries(all_global_geom, crs=crs_med).total_bounds

# Use the same buffer you used for the global figure (adjust if you changed it)
buffer = 1.2e6

# Plot
fig, ax = plt.subplots(figsize=(10, 8))

world_p.plot(ax=ax, linewidth=0.4, edgecolor="white", color="lightgray", zorder=1)

imports_path_p.plot(ax=ax, linewidth=1.0, color="red", alpha=0.25, zorder=3)
exports_path_p.plot(ax=ax, linewidth=1.0, color="dodgerblue", alpha=0.25, zorder=4)

imports_p.plot(ax=ax, color="red", markersize=8, zorder=5)
exports_p.plot(ax=ax, color="dodgerblue", markersize=8, zorder=6)

legend_elements = [
    Line2D([0], [0], marker="o", color="w", label="China imports barycenter (partners)",
           markerfacecolor="red", markersize=6),
    Line2D([0], [0], marker="o", color="w", label="China exports barycenter (partners)",
           markerfacecolor="dodgerblue", markersize=6),
]
ax.legend(handles=legend_elements, loc="lower left", frameon=False, fontsize=9)

# Decade labels in black + endpoints
years = dfp["year"].astype(int).tolist()
ymin, ymax = min(years), max(years)
label_years = set(range((ymin // 10) * 10, ymax + 1, 10))
label_years.update([ymin, ymax])

dx, dy = 35000, 35000

def label_decades(gdf_sorted):
    for _, r in gdf_sorted.iterrows():
        y = int(r["year"])
        if y in label_years:
            x0, y0 = r.geometry.x, r.geometry.y
            ax.text(x0 + dx, y0 + dy, str(y), fontsize=8, color="black", alpha=0.9, zorder=10)

label_decades(imports_p.sort_values("year"))
label_decades(exports_p.sort_values("year"))

# Apply SAME map window as the global map
ax.set_xlim(minx - buffer, maxx + buffer)
ax.set_ylim(miny - buffer, maxy + buffer)

ax.set_axis_off()
plt.tight_layout()
plt.savefig(OUT_PNG, dpi=300, bbox_inches="tight")
plt.savefig(OUT_PDF, bbox_inches="tight")
plt.close(fig)

print(f"Saved: {OUT_PNG}")
print(f"Saved: {OUT_PDF}")


Validated: CHN appears in S2_1977.parquet (exporter count=17285, importer count=3335).
Wrote: C:\Python\trade\geo\barycenter\barycenter_china_trade_1977_2022.csv | Years: 46
Saved: C:\Python\trade\geo\barycenter\barycenter_china_trade_mediterranean_zoom.png
Saved: C:\Python\trade\geo\barycenter\barycenter_china_trade_mediterranean_zoom.pdf


In [13]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString, box
from pyproj import CRS

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
GEO_DIR = r"C:\Python\trade\geo"
SHAPEFILE = os.path.join(GEO_DIR, r"ne_110m_admin_0_countries\ne_110m_admin_0_countries.shp")

BARY_DIR = os.path.join(GEO_DIR, "barycenter")
CHINA_CSV = os.path.join(BARY_DIR, "barycenter_china_trade_1977_2022.csv")

OUT_IMPORTS_PNG = os.path.join(BARY_DIR, "barycenter_china_imports.png")
OUT_IMPORTS_PDF = os.path.join(BARY_DIR, "barycenter_china_imports.pdf")

OUT_EXPORTS_PNG = os.path.join(BARY_DIR, "barycenter_china_exports.png")
OUT_EXPORTS_PDF = os.path.join(BARY_DIR, "barycenter_china_exports.pdf")

os.makedirs(BARY_DIR, exist_ok=True)

# ------------------------------------------------------------
# Load basemap and remove Antarctica
# ------------------------------------------------------------
world = gpd.read_file(SHAPEFILE)
if "ISO_A3" in world.columns:
    world = world[world["ISO_A3"] != "ATA"].copy()
if world.crs is None:
    world = world.set_crs(epsg=4326)

# ------------------------------------------------------------
# Load China barycenters (already computed)
# ------------------------------------------------------------
df = pd.read_csv(CHINA_CSV).sort_values("year")
df = df.dropna(subset=["lat_exports", "lon_exports", "lat_imports", "lon_imports"]).copy()

exports = gpd.GeoDataFrame(
    df[["year", "lat_exports", "lon_exports"]].copy(),
    geometry=[Point(xy) for xy in zip(df["lon_exports"], df["lat_exports"])],
    crs="EPSG:4326",
).sort_values("year")

imports = gpd.GeoDataFrame(
    df[["year", "lat_imports", "lon_imports"]].copy(),
    geometry=[Point(xy) for xy in zip(df["lon_imports"], df["lat_imports"])],
    crs="EPSG:4326",
).sort_values("year")

exports_path = gpd.GeoDataFrame(
    {"name": ["exports_path"]},
    geometry=[LineString(list(exports.geometry.values))],
    crs="EPSG:4326",
)

imports_path = gpd.GeoDataFrame(
    {"name": ["imports_path"]},
    geometry=[LineString(list(imports.geometry.values))],
    crs="EPSG:4326",
)

# ------------------------------------------------------------
# Projection + fixed window (Central Asia; include south)
# ------------------------------------------------------------
crs_ca = CRS.from_proj4("+proj=aeqd +lat_0=40 +lon_0=80 +datum=WGS84 +units=m +no_defs")

# Window: Eastern Mediterranean -> West China; include more south to avoid chopping
lon_min, lon_max = 15, 115
lat_min, lat_max = 5, 60

bbox_ll = gpd.GeoSeries([box(lon_min, lat_min, lon_max, lat_max)], crs="EPSG:4326")
bbox_p = bbox_ll.to_crs(crs_ca).iloc[0]
xmin, ymin, xmax, ymax = bbox_p.bounds

# Project layers
world_p = world.to_crs(crs_ca)
exports_p = exports.to_crs(crs_ca)
imports_p = imports.to_crs(crs_ca)
exports_path_p = exports_path.to_crs(crs_ca)
imports_path_p = imports_path.to_crs(crs_ca)

# ------------------------------------------------------------
# Labels: small, along trajectory
# ------------------------------------------------------------
LABEL_EVERY = 5  # label every 5 years; set to 10 if too dense

years = df["year"].astype(int).tolist()
ymin_year, ymax_year = min(years), max(years)

label_years = set(range(ymin_year, ymax_year + 1, LABEL_EVERY))
label_years.update([ymin_year, ymax_year])  # force endpoints

def add_year_labels(ax, gdf_sorted, dx=25000, dy=25000, fontsize=6):
    for _, r in gdf_sorted.iterrows():
        y = int(r["year"])
        if y in label_years:
            x0, y0 = r.geometry.x, r.geometry.y
            ax.text(
                x0 + dx, y0 + dy, str(y),
                fontsize=fontsize, color="black", alpha=0.85, zorder=10
            )

# ------------------------------------------------------------
# Render function: single-map output with bottom caption-style label
# ------------------------------------------------------------
def render_single_map(points_p, path_p, color, footer_label, out_png, out_pdf):
    fig, ax = plt.subplots(figsize=(10, 6))  # taller aspect

    # Basemap
    world_p.plot(ax=ax, linewidth=0.4, edgecolor="white", color="lightgray", zorder=1)

    # Path + points
    path_p.plot(ax=ax, linewidth=1.1, color=color, alpha=0.28, zorder=3)
    points_p.plot(ax=ax, color=color, markersize=8, zorder=4)

    # Year labels
    add_year_labels(ax, points_p.sort_values("year"), dx=25000, dy=25000, fontsize=6)

    # Fixed extent
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    # Bottom caption-style label (like your previous maps)
    ax.text(
        0.02, 0.02, footer_label,
        transform=ax.transAxes,
        fontsize=9,
        color="black",
        ha="left",
        va="bottom"
    )

    ax.set_axis_off()
    plt.tight_layout()

    # Save (overwrite)
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved: {out_png}")
    print(f"Saved: {out_pdf}")

# ------------------------------------------------------------
# Produce two separate files
# ------------------------------------------------------------
render_single_map(
    imports_p,
    imports_path_p,
    "red",
    "China imports barycenter",
    OUT_IMPORTS_PNG,
    OUT_IMPORTS_PDF
)

render_single_map(
    exports_p,
    exports_path_p,
    "dodgerblue",
    "China exports barycenter",
    OUT_EXPORTS_PNG,
    OUT_EXPORTS_PDF
)


Saved: C:\Python\trade\geo\barycenter\barycenter_china_imports.png
Saved: C:\Python\trade\geo\barycenter\barycenter_china_imports.pdf
Saved: C:\Python\trade\geo\barycenter\barycenter_china_exports.png
Saved: C:\Python\trade\geo\barycenter\barycenter_china_exports.pdf


**Muliple Country Barycenteres**

In [9]:
import os
import duckdb
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# CONFIG (edit TRADE_DIR if needed; add ISO3 codes in FOCUS_CODES)
# ------------------------------------------------------------
GEO_DIR = r"C:\Python\trade\geo"
CENTROIDS_CSV = os.path.join(GEO_DIR, "country_centroids_augmented.csv")

TRADE_DIR = r"C:\Python\trade\dataverse_files"   # <-- set to where S2_YYYY.parquet lives
START_YEAR, END_YEAR = 1977, 2022

OUT_DIR = os.path.join(GEO_DIR, "barycenter")
os.makedirs(OUT_DIR, exist_ok=True)

# Add / remove ISO3 codes here (manual control)
FOCUS_CODES = ["BRA", "ARG", "MEX", "COL", "CHL", "CAN", "USA", "CHN", "JPN", "GBR","DEU", "FRA"]

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    w = weights.to_numpy(dtype=float)
    v = values.to_numpy(dtype=float)
    s = np.nansum(w)
    if s == 0 or np.isnan(s):
        return np.nan
    return float(np.nansum(v * w) / s)

def trunc_codes(codes, max_codes=50) -> str:
    codes = list(codes) if codes is not None else []
    if not codes:
        return ""
    if len(codes) <= max_codes:
        return ";".join(codes)
    return ";".join(codes[:max_codes]) + f";...(plus {len(codes)-max_codes} more)"

# ------------------------------------------------------------
# Load centroids (single source of truth)
# ------------------------------------------------------------
centroids = pd.read_csv(CENTROIDS_CSV, usecols=["code", "lat", "lon"])
centroids["code"] = centroids["code"].astype(str)
centroids = centroids.dropna(subset=["lat", "lon"]).copy()

# ------------------------------------------------------------
# DuckDB connection
# ------------------------------------------------------------
con = duckdb.connect()

# Collect rows per country, then write one file per ISO3
rows_by_country = {c: [] for c in FOCUS_CODES}

for year in range(START_YEAR, END_YEAR + 1):
    fpath = os.path.join(TRADE_DIR, f"S2_{year}.parquet")
    if not os.path.exists(fpath):
        continue

    for c in FOCUS_CODES:
        # Exports: exporter=c, partner is importer
        ex = con.execute(f"""
            SELECT importer AS partner, SUM(value_final) AS w
            FROM read_parquet('{fpath}')
            WHERE exporter = '{c}'
            GROUP BY importer
        """).df()

        # Imports: importer=c, partner is exporter
        im = con.execute(f"""
            SELECT exporter AS partner, SUM(value_final) AS w
            FROM read_parquet('{fpath}')
            WHERE importer = '{c}'
            GROUP BY exporter
        """).df()

        ex["partner"] = ex["partner"].astype(str)
        im["partner"] = im["partner"].astype(str)

        exg = ex.merge(centroids, left_on="partner", right_on="code", how="left")
        img = im.merge(centroids, left_on="partner", right_on="code", how="left")

        ex_unmatched = exg["lat"].isna() | exg["lon"].isna()
        im_unmatched = img["lat"].isna() | img["lon"].isna()

        exm = exg.loc[~ex_unmatched].copy()
        imm = img.loc[~im_unmatched].copy()

        total_exports = float(exm["w"].sum()) if len(exm) else 0.0
        total_imports = float(imm["w"].sum()) if len(imm) else 0.0

        lat_exports = weighted_mean(exm["lat"], exm["w"]) if len(exm) else np.nan
        lon_exports = weighted_mean(exm["lon"], exm["w"]) if len(exm) else np.nan

        lat_imports = weighted_mean(imm["lat"], imm["w"]) if len(imm) else np.nan
        lon_imports = weighted_mean(imm["lon"], imm["w"]) if len(imm) else np.nan

        rows_by_country[c].append({
            "year": year,

            "lat_exports": lat_exports,
            "lon_exports": lon_exports,
            "lat_imports": lat_imports,
            "lon_imports": lon_imports,

            "total_exports_value_final_matched": total_exports,
            "total_imports_value_final_matched": total_imports,

            "n_export_partners_total": int(len(exg)),
            "n_import_partners_total": int(len(img)),
            "n_export_partners_unmatched": int(ex_unmatched.sum()),
            "n_import_partners_unmatched": int(im_unmatched.sum()),

            "unmatched_export_partners_sample": trunc_codes(exg.loc[ex_unmatched, "partner"].unique()),
            "unmatched_import_partners_sample": trunc_codes(img.loc[im_unmatched, "partner"].unique()),
        })

# Write one CSV per ISO3 (overwrite)
for c, rows in rows_by_country.items():
    out = pd.DataFrame(rows).sort_values("year")

    out_file = os.path.join(OUT_DIR, f"barycenter_{c}_1977_2022.csv")
    out.to_csv(out_file, index=False)

    if len(out):
        print(f"Wrote: {out_file} | years {int(out['year'].min())}-{int(out['year'].max())} | rows={len(out)}")
    else:
        print(f"Wrote: {out_file} | no rows (code may be absent in trade files)")

print("\nTo add more countries, edit FOCUS_CODES at the top (ISO3 codes).")


Wrote: C:\Python\trade\geo\barycenter\barycenter_BRA_1977_2022.csv | years 1977-2022 | rows=46
Wrote: C:\Python\trade\geo\barycenter\barycenter_ARG_1977_2022.csv | years 1977-2022 | rows=46
Wrote: C:\Python\trade\geo\barycenter\barycenter_MEX_1977_2022.csv | years 1977-2022 | rows=46
Wrote: C:\Python\trade\geo\barycenter\barycenter_COL_1977_2022.csv | years 1977-2022 | rows=46
Wrote: C:\Python\trade\geo\barycenter\barycenter_CHL_1977_2022.csv | years 1977-2022 | rows=46
Wrote: C:\Python\trade\geo\barycenter\barycenter_CAN_1977_2022.csv | years 1977-2022 | rows=46
Wrote: C:\Python\trade\geo\barycenter\barycenter_USA_1977_2022.csv | years 1977-2022 | rows=46
Wrote: C:\Python\trade\geo\barycenter\barycenter_CHN_1977_2022.csv | years 1977-2022 | rows=46
Wrote: C:\Python\trade\geo\barycenter\barycenter_JPN_1977_2022.csv | years 1977-2022 | rows=46
Wrote: C:\Python\trade\geo\barycenter\barycenter_GBR_1977_2022.csv | years 1977-2022 | rows=46
Wrote: C:\Python\trade\geo\barycenter\barycenter_D

**MAPS**

In [17]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString, box
from pyproj import CRS
from matplotlib.lines import Line2D

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
GEO_DIR = r"C:\Python\trade\geo"
SHAPEFILE = os.path.join(GEO_DIR, r"ne_110m_admin_0_countries\ne_110m_admin_0_countries.shp")
BARY_DIR = os.path.join(GEO_DIR, "barycenter")

COUNTRIES = ["BRA", "ARG", "MEX", "COL", "CHL", "CAN", "JPN", "GBR", "DEU", "FRA"]
COUNTRY_NAMES = {
    "BRA": "Brazil", "ARG": "Argentina", "MEX": "Mexico", "COL": "Colombia", "CHL": "Chile",
    "CAN": "Canada", "JPN": "Japan", "GBR": "United Kingdom", "DEU": "Germany", "FRA": "France"
}

LABEL_YEARS = {1977, 1980, 1985, 1994, 2001, 2009, 2018, 2023}
LABEL_FONTSIZE = 5

OUT_PNG = os.path.join(BARY_DIR, "barycenter_panel_exploratory_mercator.png")
OUT_PDF = os.path.join(BARY_DIR, "barycenter_panel_exploratory_mercator.pdf")

# Mercator
CRS_MERC = CRS.from_epsg(3857)

# Global frame (Mercator-safe latitudes)
GLOBAL_LON_MIN, GLOBAL_LON_MAX = -180, 180
GLOBAL_LAT_MIN, GLOBAL_LAT_MAX = -60, 85  # avoid near-pole blow-up

# Precompute projected bounds for consistent axes across panels
bbox_ll = gpd.GeoSeries([box(GLOBAL_LON_MIN, GLOBAL_LAT_MIN, GLOBAL_LON_MAX, GLOBAL_LAT_MAX)], crs="EPSG:4326")
bbox_p = bbox_ll.to_crs(CRS_MERC).iloc[0]
XMIN, YMIN, XMAX, YMAX = bbox_p.bounds

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def add_year_labels(ax, gdf_sorted, dx=40000, dy=40000, fontsize=LABEL_FONTSIZE):
    for _, r in gdf_sorted.iterrows():
        y = int(r["year"])
        if y in LABEL_YEARS:
            x0, y0 = r.geometry.x, r.geometry.y
            ax.text(
                x0 + dx, y0 + dy, str(y),
                fontsize=fontsize, color="black", alpha=0.8, zorder=10
            )

def load_country_barycenters(iso3: str) -> pd.DataFrame:
    fp = os.path.join(BARY_DIR, f"barycenter_{iso3}_1977_2022.csv")
    if not os.path.exists(fp):
        raise FileNotFoundError(f"Missing file for {iso3}: {fp}")

    df = pd.read_csv(fp).sort_values("year")
    needed = ["year", "lat_exports", "lon_exports", "lat_imports", "lon_imports"]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"{iso3} file missing columns: {missing}")

    return df.dropna(subset=["lat_exports", "lon_exports", "lat_imports", "lon_imports"]).copy()

def plot_country(ax, world_p, iso3: str):
    df = load_country_barycenters(iso3)

    exports = gpd.GeoDataFrame(
        df[["year", "lat_exports", "lon_exports"]].copy(),
        geometry=[Point(xy) for xy in zip(df["lon_exports"], df["lat_exports"])],
        crs="EPSG:4326",
    ).to_crs(CRS_MERC).sort_values("year")

    imports = gpd.GeoDataFrame(
        df[["year", "lat_imports", "lon_imports"]].copy(),
        geometry=[Point(xy) for xy in zip(df["lon_imports"], df["lat_imports"])],
        crs="EPSG:4326",
    ).to_crs(CRS_MERC).sort_values("year")

    exports_path = gpd.GeoDataFrame(geometry=[LineString(exports.geometry.tolist())], crs=CRS_MERC)
    imports_path = gpd.GeoDataFrame(geometry=[LineString(imports.geometry.tolist())], crs=CRS_MERC)

    # Basemap
    world_p.plot(ax=ax, linewidth=0.25, edgecolor="white", color="lightgray", zorder=1)

    # Trajectories
    imports_path.plot(ax=ax, linewidth=0.9, color="red", alpha=0.25, zorder=3)
    exports_path.plot(ax=ax, linewidth=0.9, color="dodgerblue", alpha=0.25, zorder=4)

    # Points
    imports.plot(ax=ax, color="red", markersize=4, zorder=5)
    exports.plot(ax=ax, color="dodgerblue", markersize=4, zorder=6)

    # Labels
    add_year_labels(ax, imports, dx=40000, dy=40000, fontsize=LABEL_FONTSIZE)
    add_year_labels(ax, exports, dx=40000, dy=40000, fontsize=LABEL_FONTSIZE)

    # Consistent global extent
    ax.set_xlim(XMIN, XMAX)
    ax.set_ylim(YMIN, YMAX)

    ax.set_axis_off()
    ax.set_title(f"{COUNTRY_NAMES.get(iso3, iso3)} ({iso3})", fontsize=10)

# ------------------------------------------------------------
# Load basemap (remove Antarctica) and project once
# ------------------------------------------------------------
world = gpd.read_file(SHAPEFILE)
if "ISO_A3" in world.columns:
    world = world[world["ISO_A3"] != "ATA"].copy()
if world.crs is None:
    world = world.set_crs(epsg=4326)

# Clip to Mercator-safe latitudes for cleaner rendering
clip_poly = gpd.GeoSeries([box(-180, GLOBAL_LAT_MIN, 180, GLOBAL_LAT_MAX)], crs="EPSG:4326")
world = gpd.clip(world, clip_poly)

world_p = world.to_crs(CRS_MERC)

# ------------------------------------------------------------
# Panel layout: 5 rows x 2 cols (10 countries)
# ------------------------------------------------------------
fig, axes = plt.subplots(5, 2, figsize=(12, 18), constrained_layout=True)
axes = axes.flatten()

for ax, iso3 in zip(axes, COUNTRIES):
    plot_country(ax, world_p, iso3)

# Shared legend (bottom)
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='Imports barycenter',
           markerfacecolor='red', markersize=6),
    Line2D([0], [0], marker='o', color='w', label='Exports barycenter',
           markerfacecolor='dodgerblue', markersize=6),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=2, frameon=False, fontsize=10)

os.makedirs(BARY_DIR, exist_ok=True)
plt.savefig(OUT_PNG, dpi=300, bbox_inches="tight")
plt.savefig(OUT_PDF, bbox_inches="tight")
plt.close(fig)

print(f"Saved: {OUT_PNG}")
print(f"Saved: {OUT_PDF}")


Saved: C:\Python\trade\geo\barycenter\barycenter_panel_exploratory_mercator.png
Saved: C:\Python\trade\geo\barycenter\barycenter_panel_exploratory_mercator.pdf


**MAPS**

In [18]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString, box
from pyproj import CRS

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
GEO_DIR = r"C:\Python\trade\geo"
SHAPEFILE = os.path.join(GEO_DIR, r"ne_110m_admin_0_countries\ne_110m_admin_0_countries.shp")
BARY_DIR = os.path.join(GEO_DIR, "barycenter")

COUNTRIES = ["BRA","ARG","CHL","COL","MEX","CAN","JPN","GBR","DEU","FRA"]

COUNTRY_NAMES = {
    "BRA": "Brazil",
    "ARG": "Argentina",
    "CHL": "Chile",
    "COL": "Colombia",
    "MEX": "Mexico",
    "CAN": "Canada",
    "JPN": "Japan",
    "GBR": "United Kingdom",
    "DEU": "Germany",
    "FRA": "France",
}

# Labels (updated): use 2022 (not 2023)
LABEL_YEARS = {1977, 1981, 1985, 1988, 1994, 1997, 2001, 2009, 2018, 2022}
EMPH_YEARS = {1977, 2022}  # bold + italic

# Visuals
BASEMAP_FACE = "lightgray"
BASEMAP_EDGE = "white"
BASEMAP_LW = 0.35
PATH_LW = 1.1
PATH_ALPHA = 0.28
POINT_SIZE = 8

# Smaller year labels
LABEL_FONTSIZE = 4
LABEL_ALPHA = 0.8

# Output filename format: barycenter_<ISO3>_exports/imports
OUT_FMT_PNG = os.path.join(BARY_DIR, "barycenter_{iso3}_{flow}.png")
OUT_FMT_PDF = os.path.join(BARY_DIR, "barycenter_{iso3}_{flow}.pdf")

os.makedirs(BARY_DIR, exist_ok=True)

# ------------------------------------------------------------
# CRS helpers
# ------------------------------------------------------------
def aeqd(lat0, lon0) -> CRS:
    return CRS.from_proj4(f"+proj=aeqd +lat_0={lat0} +lon_0={lon0} +datum=WGS84 +units=m +no_defs")

# ------------------------------------------------------------
# Projection + extent rules (lon/lat)
# ------------------------------------------------------------
# 1) Argentina, Brazil, Chile:
#    Center North Africa; N=southern France (~43N), S=equatorial Africa (0),
#    W=eastern tip of Brazil (~-35), E=Iraq (~44E)
CRS_N_AFRICA = aeqd(20, 10)
BBOX_N_AFRICA = (-35, 0, 44, 43)

# 2) Colombia:
#    N=UK (~58N), S=Ecuador (~-5), W=Yucatan (~-92), E=France (~2)
CRS_COL = aeqd(15, -40)
BBOX_COL = (-92, -5, 2, 58)

# 3) Mexico:
#    Projection centered around NY (ok)
#    Extent: N=southern Canada (~49N), S=southern Mexico (~14N),
#            W=California (~-125), E=Spain (~3E)
CRS_MEX = aeqd(40.7, -74.0)
BBOX_MEX = (-125, 14, 3, 49)

# 4) Canada: use same projection and extent as Mexico
CRS_CAN = CRS_MEX
BBOX_CAN = BBOX_MEX

# 5) Japan:
#    Center Egypt; N=southern Europe (~45N), S=Yemen (~12N),
#    W=Canary Islands (~-16), E=Pakistan (~75E)
CRS_JPN = aeqd(26, 30)
BBOX_JPN = (-16, 12, 75, 45)

# 6) European countries: use Mediterranean projection + extent from USA logic
CRS_MED = aeqd(35, 18)
GLOBAL_BARY_FILE = os.path.join(BARY_DIR, "barycenter_imports_exports_1977_2022.csv")
EU_ISOS = {"GBR", "DEU", "FRA"}

COUNTRY_SETTINGS = {
    "ARG": {"crs": CRS_N_AFRICA, "bbox": BBOX_N_AFRICA},
    "BRA": {"crs": CRS_N_AFRICA, "bbox": BBOX_N_AFRICA},
    "CHL": {"crs": CRS_N_AFRICA, "bbox": BBOX_N_AFRICA},

    "COL": {"crs": CRS_COL,      "bbox": BBOX_COL},

    "MEX": {"crs": CRS_MEX,      "bbox": BBOX_MEX},
    "CAN": {"crs": CRS_CAN,      "bbox": BBOX_CAN},

    "JPN": {"crs": CRS_JPN,      "bbox": BBOX_JPN},

    # EU: bbox ignored; extent computed from global barycenter file under CRS_MED
    "GBR": {"crs": CRS_MED,      "bbox": None},
    "DEU": {"crs": CRS_MED,      "bbox": None},
    "FRA": {"crs": CRS_MED,      "bbox": None},
}

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def load_barycenter_df(iso3: str) -> pd.DataFrame:
    fp = os.path.join(BARY_DIR, f"barycenter_{iso3}_1977_2022.csv")
    if not os.path.exists(fp):
        raise FileNotFoundError(f"Missing barycenter file: {fp}")
    df = pd.read_csv(fp).sort_values("year")
    needed = ["year", "lat_exports", "lon_exports", "lat_imports", "lon_imports"]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"{iso3}: missing columns {missing} in {fp}")
    return df.dropna(subset=["lat_exports","lon_exports","lat_imports","lon_imports"]).copy()

def add_year_labels(ax, gdf_sorted, dx, dy):
    for _, r in gdf_sorted.iterrows():
        y = int(r["year"])
        if y in LABEL_YEARS:
            is_emph = y in EMPH_YEARS
            ax.text(
                r.geometry.x + dx,
                r.geometry.y + dy,
                str(y),
                fontsize=LABEL_FONTSIZE,
                color="black",
                alpha=LABEL_ALPHA,
                fontweight=("bold" if is_emph else "normal"),
                fontstyle=("italic" if is_emph else "normal"),
                zorder=10
            )

def project_bbox(crs_target: CRS, bbox_lonlat):
    lon_min, lat_min, lon_max, lat_max = bbox_lonlat
    bbox_ll = gpd.GeoSeries([box(lon_min, lat_min, lon_max, lat_max)], crs="EPSG:4326")
    bbox_p = bbox_ll.to_crs(crs_target).iloc[0]
    return bbox_p.bounds  # xmin, ymin, xmax, ymax

def compute_med_extent_from_global():
    """
    Reproduce the USA Mediterranean extent logic:
    project global import+export barycenters to CRS_MED and take bounds + buffer.
    """
    if not os.path.exists(GLOBAL_BARY_FILE):
        raise FileNotFoundError(f"Missing global barycenter file for EU extent: {GLOBAL_BARY_FILE}")

    g = pd.read_csv(GLOBAL_BARY_FILE).dropna(
        subset=["lat_exports","lon_exports","lat_imports","lon_imports"]
    )

    ge = gpd.GeoDataFrame(
        geometry=[Point(xy) for xy in zip(g["lon_exports"], g["lat_exports"])],
        crs="EPSG:4326"
    ).to_crs(CRS_MED)

    gi = gpd.GeoDataFrame(
        geometry=[Point(xy) for xy in zip(g["lon_imports"], g["lat_imports"])],
        crs="EPSG:4326"
    ).to_crs(CRS_MED)

    minx, miny, maxx, maxy = gpd.GeoSeries(
        pd.concat([ge.geometry, gi.geometry], ignore_index=True),
        crs=CRS_MED
    ).total_bounds

    buffer = 1.2e6
    return (minx - buffer, miny - buffer, maxx + buffer, maxy + buffer)

EU_XMIN, EU_YMIN, EU_XMAX, EU_YMAX = compute_med_extent_from_global()

def get_extent_for_country(iso3: str):
    s = COUNTRY_SETTINGS[iso3]
    crs = s["crs"]
    if iso3 in EU_ISOS:
        return crs, (EU_XMIN, EU_YMIN, EU_XMAX, EU_YMAX)
    xmin, ymin, xmax, ymax = project_bbox(crs, s["bbox"])
    return crs, (xmin, ymin, xmax, ymax)

def render_map_for_country_flow(world_ll, iso3: str, flow: str):
    """
    flow: "imports" or "exports"
    """
    crs, (xmin, ymin, xmax, ymax) = get_extent_for_country(iso3)
    df = load_barycenter_df(iso3)

    if flow == "imports":
        lat_col, lon_col = "lat_imports", "lon_imports"
        color = "red"
        footer = f"{COUNTRY_NAMES.get(iso3, iso3)} ({iso3}) — Imports barycenter"
    elif flow == "exports":
        lat_col, lon_col = "lat_exports", "lon_exports"
        color = "dodgerblue"
        footer = f"{COUNTRY_NAMES.get(iso3, iso3)} ({iso3}) — Exports barycenter"
    else:
        raise ValueError("flow must be 'imports' or 'exports'")

    gdf = gpd.GeoDataFrame(
        df[["year", lat_col, lon_col]].copy(),
        geometry=[Point(xy) for xy in zip(df[lon_col], df[lat_col])],
        crs="EPSG:4326"
    ).sort_values("year").to_crs(crs)

    path = gpd.GeoDataFrame(geometry=[LineString(gdf.geometry.tolist())], crs=crs)

    world_p = world_ll.to_crs(crs)

    fig, ax = plt.subplots(figsize=(10, 6))
    world_p.plot(ax=ax, linewidth=BASEMAP_LW, edgecolor=BASEMAP_EDGE, color=BASEMAP_FACE, zorder=1)

    path.plot(ax=ax, linewidth=PATH_LW, color=color, alpha=PATH_ALPHA, zorder=3)
    gdf.plot(ax=ax, color=color, markersize=POINT_SIZE, zorder=4)

    # Labels (smaller) with emphasis on 1977 and 2022
    add_year_labels(ax, gdf, dx=15000, dy=15000)

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    # Lower-left legend text
    ax.text(
        0.02, 0.02, footer,
        transform=ax.transAxes,
        fontsize=9,
        color="black",
        ha="left",
        va="bottom"
    )

    ax.set_axis_off()
    plt.tight_layout()

    out_png = OUT_FMT_PNG.format(iso3=iso3, flow=flow)
    out_pdf = OUT_FMT_PDF.format(iso3=iso3, flow=flow)

    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved: {out_png}")
    print(f"Saved: {out_pdf}")

# ------------------------------------------------------------
# Load basemap and remove Antarctica
# ------------------------------------------------------------
world = gpd.read_file(SHAPEFILE)
if "ISO_A3" in world.columns:
    world = world[world["ISO_A3"] != "ATA"].copy()
if world.crs is None:
    world = world.set_crs(epsg=4326)

# ------------------------------------------------------------
# Render: two maps per country
# ------------------------------------------------------------
for iso3 in COUNTRIES:
    render_map_for_country_flow(world, iso3, "imports")
    render_map_for_country_flow(world, iso3, "exports")


Saved: C:\Python\trade\geo\barycenter\barycenter_BRA_imports.png
Saved: C:\Python\trade\geo\barycenter\barycenter_BRA_imports.pdf
Saved: C:\Python\trade\geo\barycenter\barycenter_BRA_exports.png
Saved: C:\Python\trade\geo\barycenter\barycenter_BRA_exports.pdf
Saved: C:\Python\trade\geo\barycenter\barycenter_ARG_imports.png
Saved: C:\Python\trade\geo\barycenter\barycenter_ARG_imports.pdf
Saved: C:\Python\trade\geo\barycenter\barycenter_ARG_exports.png
Saved: C:\Python\trade\geo\barycenter\barycenter_ARG_exports.pdf
Saved: C:\Python\trade\geo\barycenter\barycenter_CHL_imports.png
Saved: C:\Python\trade\geo\barycenter\barycenter_CHL_imports.pdf
Saved: C:\Python\trade\geo\barycenter\barycenter_CHL_exports.png
Saved: C:\Python\trade\geo\barycenter\barycenter_CHL_exports.pdf
Saved: C:\Python\trade\geo\barycenter\barycenter_COL_imports.png
Saved: C:\Python\trade\geo\barycenter\barycenter_COL_imports.pdf
Saved: C:\Python\trade\geo\barycenter\barycenter_COL_exports.png
Saved: C:\Python\trade\ge